# Simulator Pertumbuhan Pengguna Internet di Indonesia (2000–2024)
## Menggunakan Metode Euler pada Model Pertumbuhan Logistik

**Mata Kuliah:** Praktikum Pemodelan dan Simulasi — Modul 7: Dynamic System Simulation

**Studi kasus:** Persentase penduduk Indonesia yang menggunakan internet.
**Sumber data:** World Bank – World Development Indicators, indikator *Individuals using the Internet (% of population)* (kode `IT.NET.USER.ZS`), sumber asli ITU, lisensi CC BY-4.0.
Tautan: https://data.worldbank.org/indicator/IT.NET.USER.ZS?locations=ID

### Model matematika
Model Pertumbuhan Logistik:

$$\frac{dP}{dt} = r\,P\left(1-\frac{P}{K}\right)$$

Diselesaikan dengan Metode Euler:

$$P_{n+1} = P_n + h\cdot r\,P_n\left(1-\frac{P_n}{K}\right)$$

| Simbol | Arti | Satuan |
|---|---|---|
| $P$ | Persentase penduduk pengguna internet (Stock) | % |
| $t$ | Waktu | tahun |
| $r$ | Laju pertumbuhan (adopsi) | per tahun |
| $K$ | Kapasitas maksimum | % ($K \le 100$) |
| $h$ | Step size Euler | tahun |

**Struktur sistem:**
- **Stock:** $P$ · **Flow masuk:** pengguna baru per tahun $= rP(1-P/K)$
- **Loop R (penguat):** makin banyak pengguna → makin banyak pengguna baru.
- **Loop B (penyeimbang):** makin dekat $P$ ke $K$ → ruang sisa $(1-P/K)$ mengecil → pertumbuhan melambat.

## 1. Import Library
- `pandas` → membaca dan mengolah data tabel
- `numpy` → perhitungan numerik
- `matplotlib` → membuat grafik

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Pengaturan tampilan grafik agar seragam
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

print("Library berhasil di-import")

## 2. Memuat Dataset
File CSV dari World Bank memiliki **4 baris keterangan** di bagian atas (sumber data dan tanggal pembaruan), sehingga 4 baris itu dilewati dengan `skiprows=4`.

In [ ]:
# Lokasi file dataset (relatif terhadap lokasi notebook)
PATH_DATA = "data/API_IT.NET.USER.ZS_DS2_en_csv_v2_62387.csv"

df_raw = pd.read_csv(PATH_DATA, skiprows=4)

print("Ukuran dataset mentah (baris, kolom):", df_raw.shape)
df_raw.head()

## 3. Memeriksa Struktur Data
Dataset mentah berformat **lebar (wide)**: satu baris = satu negara, dan setiap tahun menjadi satu kolom. Kita lihat nama-nama kolomnya.

In [ ]:
print("Kolom pertama :", df_raw.columns[:5].tolist())
print("Kolom terakhir:", df_raw.columns[-5:].tolist())
print("Jumlah negara/wilayah:", df_raw["Country Name"].nunique())

## 4. Mengambil Data Indonesia dan Mengubah ke Format Panjang (Long)
Untuk simulasi kita membutuhkan dua kolom saja: **Tahun** dan **Persen** (persentase pengguna internet). Langkahnya:
1. Ambil baris dengan kode negara `IDN`.
2. Ambil kolom-kolom yang namanya berupa angka tahun.
3. Ubah menjadi tabel dua kolom (format long).

In [ ]:
# 1. Ambil baris Indonesia
baris_idn = df_raw[df_raw["Country Code"] == "IDN"]

# 2. Ambil hanya kolom tahun (nama kolom berupa angka, misal "1990")
kolom_tahun = [c for c in df_raw.columns if c.isdigit()]

# 3. Ubah ke format long: kolom Tahun dan Persen
df = baris_idn[kolom_tahun].T.reset_index()
df.columns = ["Tahun", "Persen"]
df["Tahun"] = df["Tahun"].astype(int)
df["Persen"] = pd.to_numeric(df["Persen"], errors="coerce")

print("Ukuran data Indonesia:", df.shape)
df.head(10)

## 5. Exploratory Data Analysis (EDA)
### 5.1 Tipe data

In [ ]:
df.dtypes

### 5.2 Missing values
World Bank menyediakan kolom tahun mulai 1960, padahal internet baru tercatat di Indonesia sekitar 1990-an. Jadi wajar jika banyak tahun awal yang kosong. Yang penting dicek: **apakah ada tahun kosong di tengah rentang data yang terisi?**

In [ ]:
print("Total baris              :", len(df))
print("Jumlah nilai kosong (NaN):", df["Persen"].isnull().sum())

terisi = df.dropna()
print("Jumlah nilai terisi      :", len(terisi))
print("Tahun terisi pertama     :", terisi["Tahun"].min())
print("Tahun terisi terakhir    :", terisi["Tahun"].max())

# Tahun yang terisi (untuk melihat ada/tidaknya lubang)
print("\nDaftar tahun terisi:")
print(terisi["Tahun"].tolist())

In [ ]:
# Cek lubang di rentang 1994–2024 (setelah nilai > 0 pertama)
mulai = terisi.loc[terisi["Persen"] > 0, "Tahun"].min()
akhir = terisi["Tahun"].max()
rentang = df[(df["Tahun"] >= mulai) & (df["Tahun"] <= akhir)]

print(f"Rentang {mulai}–{akhir}: {len(rentang)} tahun")
print("Tahun kosong di rentang ini:", rentang[rentang["Persen"].isnull()]["Tahun"].tolist())

**Interpretasi:** Dari 66 kolom tahun (1960–2025), hanya 32 yang terisi. Tahun 1990 bernilai 0, lalu data berurutan **1994–2024 tanpa tahun yang kosong**. Kolom 2025 belum terisi. Jadi tidak perlu imputasi (pengisian nilai kosong).

### 5.3 Duplikasi

In [ ]:
print("Duplikasi pada kolom Tahun :", terisi["Tahun"].duplicated().sum())
print("Duplikasi pada kolom Persen:", terisi["Persen"].duplicated().sum())
print("Duplikasi baris utuh       :", terisi.duplicated().sum())

**Interpretasi:** Tidak ada duplikasi. Setiap tahun hanya muncul sekali, dan nilai persentase selalu berbeda karena terus berubah dari tahun ke tahun.

### 5.4 Statistik deskriptif

In [ ]:
terisi["Persen"].describe()

**Interpretasi:** Nilai minimum 0% (tahun 1990) dan maksimum 72,78% (tahun 2024). Median (7,42%) jauh lebih kecil daripada rata-rata (18,80%) karena sebagian besar tahun awal nilainya masih kecil, lalu melonjak di tahun-tahun akhir — ciri data yang tumbuh cepat belakangan.

### 5.5 Visualisasi data asli

In [ ]:
plt.figure()
plt.plot(terisi["Tahun"], terisi["Persen"], marker="o")
plt.axvline(2000, color="gray", linestyle="--", label="Awal rentang simulasi (2000)")
plt.title("Persentase Penduduk Indonesia Pengguna Internet")
plt.xlabel("Tahun")
plt.ylabel("Pengguna internet (% penduduk)")
plt.legend()
plt.show()

**Interpretasi:** Kurva berbentuk **S (sigmoid)**: naik pelan sampai sekitar 2014, melonjak tajam 2015–2021, lalu mulai melandai 2022–2024. Pola ini sesuai dengan model logistik.

### 5.6 Pertambahan per tahun
Pertambahan tahunan $\Delta P$ adalah pendekatan dari laju $dP/dt$. Pada model logistik, laju ini **naik lalu turun** (puncaknya di titik belok, saat $P \approx K/2$).

In [ ]:
terisi = terisi.copy()
terisi["Pertambahan"] = terisi["Persen"].diff()

plt.figure()
plt.bar(terisi["Tahun"], terisi["Pertambahan"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Pertambahan Persentase Pengguna Internet per Tahun")
plt.xlabel("Tahun")
plt.ylabel("Pertambahan (poin persen)")
plt.show()

# Tahun dengan pertambahan terbesar dan tahun dengan pertambahan negatif
idx_max = terisi["Pertambahan"].idxmax()
print("Pertambahan terbesar:", terisi.loc[idx_max, "Tahun"], "=",
      round(terisi.loc[idx_max, "Pertambahan"], 2), "poin persen")
print("Tahun dengan pertambahan negatif:")
print(terisi[terisi["Pertambahan"] < 0][["Tahun", "Persen", "Pertambahan"]])

**Interpretasi:** Pertambahan tahunan kecil di awal, membesar dan mencapai puncak **8,38 poin persen pada 2021**, lalu mengecil (4,38 → 2,72 → 3,57 poin pada 2022–2024). Pola naik-lalu-turun ini menandakan data sudah melewati titik belok kurva logistik: loop penguat (R) dominan sebelum 2021, loop penyeimbang (B) mulai dominan sesudahnya. Satu-satunya pertambahan negatif terjadi pada **2009** (turun ±1 poin persen).

## 6. Pembersihan Data dan Penentuan Data Simulasi
Keputusan pembersihan:
1. **Hapus nilai kosong (NaN)** — tahun sebelum data tercatat.
2. **Gunakan rentang 2000–2024.** Sebelum 2000 nilainya masih di bawah 1% dan pertumbuhannya tidak teratur, sehingga kurang cocok sebagai nilai awal simulasi.
3. **Nilai 2009 tetap dipakai.** Penurunan 2008→2009 kemungkinan akibat perubahan metode pengukuran; data tidak diubah, cukup dicatat sebagai keterbatasan.

Data bersih disimpan ke `data/internet_indonesia_2000_2024.csv` agar bisa dipakai ulang oleh aplikasi Streamlit nanti.

In [ ]:
TAHUN_AWAL, TAHUN_AKHIR = 2000, 2024

df_sim = terisi[(terisi["Tahun"] >= TAHUN_AWAL) & (terisi["Tahun"] <= TAHUN_AKHIR)]
df_sim = df_sim[["Tahun", "Persen"]].reset_index(drop=True)

print("Jumlah data simulasi:", len(df_sim))
print("Nilai kosong        :", df_sim["Persen"].isnull().sum())
print("Nilai awal P0 (2000):", round(df_sim["Persen"].iloc[0], 4), "%")
print("Nilai akhir (2024)  :", round(df_sim["Persen"].iloc[-1], 4), "%")

df_sim.to_csv("data/internet_indonesia_2000_2024.csv", index=False)
print("\nData bersih disimpan ke data/internet_indonesia_2000_2024.csv")
df_sim

---
# TAHAP 3 — Implementasi Simulasi Metode Euler

## 7. Definisi Persamaan Diferensial
Fungsi `laju_logistik` menghitung **laju perubahan** $dP/dt = rP(1-P/K)$, yaitu besar Flow (pengguna baru per tahun) untuk nilai $P$ tertentu.

In [ ]:
def laju_logistik(P, r, K):
    """Laju perubahan dP/dt pada model logistik (satuan: % per tahun)."""
    return r * P * (1 - P / K)

# Contoh: jika P = 50%, r = 0.3, K = 100 → laju = 0.3 * 50 * 0.5 = 7.5 % per tahun
print("Contoh laju:", laju_logistik(50, 0.3, 100), "% per tahun")

## 8. Implementasi Metode Euler
Rumus iterasi: $P_{n+1} = P_n + h \cdot f(P_n)$

Fungsi `euler_logistik` mengulang rumus tersebut dari tahun awal sampai tahun akhir dengan langkah `h`, lalu mengembalikan array waktu `t` dan nilai `P`.

In [ ]:
def euler_logistik(P0, r, K, t0, t_akhir, h):
    """Simulasi model logistik dengan Metode Euler."""
    n_langkah = int(round((t_akhir - t0) / h))      # jumlah iterasi
    t = t0 + h * np.arange(n_langkah + 1)          # titik-titik waktu
    P = np.zeros(n_langkah + 1)
    P[0] = P0                                       # nilai awal (Initial State)

    for i in range(n_langkah):                      # LOOP SIMULASI
        P[i + 1] = P[i] + h * laju_logistik(P[i], r, K)

    return t, P


def ambil_tahunan(t, P, h):
    """Ambil nilai P pada tahun bulat saja (agar bisa dibandingkan dengan data tahunan)."""
    lompat = int(round(1 / h))
    return t[::lompat], P[::lompat]

## 9. Inisialisasi Nilai Awal dan Data Pembanding

In [ ]:
tahun = df_sim["Tahun"].values
data_asli = df_sim["Persen"].values

P0 = data_asli[0]               # nilai awal Stock = data tahun 2000
t0, t_akhir = tahun[0], tahun[-1]

print(f"P0 = {P0:.4f} % (tahun {t0})")
print(f"Rentang simulasi: {t0}–{t_akhir} ({len(tahun)} titik data)")

## 10. Fungsi Evaluasi Model
Untuk mengukur seberapa dekat garis simulasi dengan data asli:
- **RMSE** (*Root Mean Square Error*) — rata-rata besar galat, satuan poin persen. Galat besar diberi bobot lebih.
- **MAE** (*Mean Absolute Error*) — rata-rata selisih mutlak, satuan poin persen.
- **MAPE** (*Mean Absolute Percentage Error*) — galat relatif terhadap nilai data, dalam %.

Semakin kecil nilainya, semakin baik.

In [ ]:
def evaluasi(asli, simulasi):
    galat = simulasi - asli
    rmse = np.sqrt(np.mean(galat ** 2))
    mae = np.mean(np.abs(galat))
    mape = np.mean(np.abs(galat / asli)) * 100
    return rmse, mae, mape

## 11. Simulasi Awal dengan Parameter Tebakan
Sebelum kalibrasi, kita coba dulu parameter tebakan (mirip tutorial modul):
- $K$ = 1,2 × nilai data tertinggi
- $r$ = 0,3 per tahun

In [ ]:
r_tebak = 0.3
K_tebak = 1.2 * data_asli.max()

t, P = euler_logistik(P0, r_tebak, K_tebak, t0, t_akhir, h=1)
rmse, mae, mape = evaluasi(data_asli, P)

print(f"Parameter tebakan: r = {r_tebak}, K = {K_tebak:.2f}")
print(f"RMSE = {rmse:.3f} | MAE = {mae:.3f} | MAPE = {mape:.2f} %")

plt.figure()
plt.scatter(tahun, data_asli, label="Data asli")
plt.plot(t, P, color="red", label="Simulasi Euler (tebakan)")
plt.title("Simulasi Awal dengan Parameter Tebakan")
plt.xlabel("Tahun"); plt.ylabel("Pengguna internet (% penduduk)")
plt.legend(); plt.show()

**Interpretasi:** Dengan parameter tebakan ($r=0{,}3$, $K=87{,}34$) kurva simulasi naik terlalu cepat setelah 2010 dan selalu di atas data. RMSE 7,209 poin persen dan MAPE 22,78% — masih jauh dari data. Parameter perlu dikalibrasi.

## 12. Kalibrasi Parameter (Tuning r dan K)
Parameter tebakan belum tentu yang terbaik. Kita mencoba **banyak kombinasi** $r$ dan $K$ (*grid search*), menjalankan Euler untuk setiap kombinasi, lalu memilih kombinasi dengan **RMSE terkecil**.

- $r$ dicoba dari 0,050 sampai 0,600 (langkah 0,001)
- $K$ dicoba dari nilai data tertinggi sampai 100 (langkah 0,1), karena data berupa persen

Agar cepat, semua kombinasi dihitung **bersamaan** memakai array NumPy. Rumus yang dipakai tetap rumus Euler yang sama ($h = 1$).

In [ ]:
r_grid = np.arange(0.050, 0.6005, 0.001)
K_grid = np.arange(np.ceil(data_asli.max()), 100.0001, 0.1)
R, KK = np.meshgrid(r_grid, K_grid)        # tabel semua kombinasi r dan K

P_grid = np.full(R.shape, P0)              # semua kombinasi mulai dari P0
total_galat2 = np.zeros(R.shape)

for i in range(1, len(data_asli)):         # Euler h = 1, satu langkah = satu tahun
    P_grid = P_grid + 1 * R * P_grid * (1 - P_grid / KK)
    total_galat2 += (P_grid - data_asli[i]) ** 2

rmse_grid = np.sqrt(total_galat2 / len(data_asli))
idx = np.unravel_index(np.argmin(rmse_grid), rmse_grid.shape)
r_opt, K_opt = R[idx], KK[idx]

print(f"Jumlah kombinasi dicoba: {R.size:,}")
print(f"Parameter terbaik → r = {r_opt:.3f} per tahun, K = {K_opt:.1f} %")
print(f"RMSE terkecil      → {rmse_grid[idx]:.3f} poin persen")

### 12.1 Peta RMSE
Warna gelap = RMSE kecil (cocok). Titik merah = parameter terbaik. Peta ini menunjukkan seberapa "pasti" nilai $r$ dan $K$ yang ditemukan.

In [ ]:
plt.figure(figsize=(9, 5))
cs = plt.contourf(R, KK, rmse_grid, levels=np.linspace(0, 15, 31), cmap="viridis_r", extend="max")
plt.colorbar(cs, label="RMSE (poin persen)")
plt.scatter(r_opt, K_opt, color="red", zorder=5, label=f"Terbaik: r={r_opt:.3f}, K={K_opt:.1f}")
plt.xlabel("r (per tahun)"); plt.ylabel("K (%)")
plt.title("Peta RMSE untuk Setiap Kombinasi r dan K")
plt.legend(); plt.grid(False); plt.show()

**Interpretasi:** Lembah warna terang di peta berbentuk **pita tegak**: nilai $r$ terkunci cukup sempit di sekitar 0,26, sedangkan $K$ dari ±73 sampai 100 memberi RMSE yang tidak jauh berbeda. Artinya **data sudah cukup untuk menentukan laju pertumbuhan $r$, tetapi belum cukup untuk menentukan batas jenuh $K$** — karena kurva Indonesia belum benar-benar mendatar. Ini keterbatasan penting yang dicatat di laporan.

## 13. Simulasi Baseline (Parameter Terkalibrasi) dan Penyimpanan Hasil ke DataFrame

In [ ]:
t_base, P_base = euler_logistik(P0, r_opt, K_opt, t0, t_akhir, h=1)

hasil_sim = pd.DataFrame({
    "Tahun": tahun,
    "Data_Asli": data_asli,
    "Simulasi_Euler": P_base,
})
hasil_sim["Selisih"] = hasil_sim["Simulasi_Euler"] - hasil_sim["Data_Asli"]
hasil_sim["Laju_dPdt"] = laju_logistik(hasil_sim["Simulasi_Euler"], r_opt, K_opt)

hasil_sim.to_csv("data/hasil_simulasi_baseline.csv", index=False)
print("Hasil simulasi disimpan ke data/hasil_simulasi_baseline.csv")
hasil_sim.round(3)

## 14. Evaluasi Model Baseline

In [ ]:
rmse, mae, mape = evaluasi(data_asli, P_base)
print("=== EVALUASI MODEL BASELINE (Euler, h = 1) ===")
print(f"r    = {r_opt:.3f} per tahun")
print(f"K    = {K_opt:.1f} %")
print(f"RMSE = {rmse:.3f} poin persen")
print(f"MAE  = {mae:.3f} poin persen")
print(f"MAPE = {mape:.2f} %")

# MAPE sangat dipengaruhi tahun-tahun awal yang nilainya kecil, jadi dihitung juga per periode
for a, b in [(2000, 2009), (2010, 2024)]:
    m = (tahun >= a) & (tahun <= b)
    print(f"MAPE {a}–{b}: {evaluasi(data_asli[m], P_base[m])[2]:.2f} %")

**Interpretasi:** Dengan $r = 0{,}261$ dan $K = 96{,}9$, RMSE turun dari 7,209 (tebakan) menjadi **1,726 poin persen** dan MAE 1,439 poin persen. MAPE keseluruhan 12,60% terlihat besar karena tahun 2000–2009 nilainya kecil (di bawah 8%), sehingga selisih kecil pun menjadi persentase besar (MAPE 20,24%). Pada 2010–2024 MAPE hanya 7,51%.

## 15. Visualisasi: Data Asli vs Simulasi

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

ax[0].scatter(tahun, data_asli, label="Data asli (World Bank)")
ax[0].plot(t_base, P_base, color="red", label=f"Simulasi Euler (r={r_opt:.3f}, K={K_opt:.1f})")
ax[0].axhline(K_opt, color="gray", linestyle="--", label="Kapasitas maksimum K")
ax[0].axhline(K_opt / 2, color="gray", linestyle=":", label="K/2 (titik belok)")
ax[0].set_ylabel("Pengguna internet (% penduduk)")
ax[0].set_title("Perbandingan Data Asli dan Simulasi Model Logistik")
ax[0].legend()

ax[1].bar(tahun, hasil_sim["Selisih"], color="orange")
ax[1].axhline(0, color="black", linewidth=0.8)
ax[1].set_xlabel("Tahun"); ax[1].set_ylabel("Selisih (sim − asli)")
ax[1].set_title("Selisih Simulasi terhadap Data Asli")
plt.tight_layout(); plt.show()

**Interpretasi grafik:**
- Garis simulasi mengikuti pola S data asli: lambat di awal, cepat di tengah, melambat di akhir.
- Selisih **tidak acak**, tetapi bergelombang: simulasi sedikit di bawah data pada 2001–2012, di atas data 2013–2017, di bawah lagi 2019–2022. Pola ini menunjukkan data asli tidak tumbuh semulus model — ada faktor luar yang tidak dimodelkan (misalnya lonjakan pengguna smartphone, pandemi 2020–2021).
- Selisih terbesar ±3,5 poin persen (2021) — data naik lebih cepat dari model.
- Data 2024 sudah berada di sekitar 73%, masih di atas $K/2$, sehingga sistem sedang di **fase transisi (melambat)** menuju $K$.

## 16. Pengaruh Step Size h (h = 1 vs h = 0,1)
Dengan $r$ dan $K$ yang sama, kita bandingkan Euler $h=1$ dan $h=0,1$ terhadap:
1. **Solusi eksak** model logistik: $P(t) = \dfrac{K}{1 + \frac{K-P_0}{P_0}e^{-r(t-t_0)}}$ — untuk melihat **galat numerik** Euler.
2. **SciPy `solve_ivp`** — solver profesional (sesuai tujuan modul) sebagai pembanding.
3. **Data asli** — untuk melihat kecocokan dengan kenyataan.

In [ ]:
from scipy.integrate import solve_ivp

def solusi_eksak(t, P0, r, K, t0):
    return K / (1 + (K - P0) / P0 * np.exp(-r * (t - t0)))

# Euler h = 1 dan h = 0.1
t1, P1 = euler_logistik(P0, r_opt, K_opt, t0, t_akhir, h=1)
t01, P01_halus = euler_logistik(P0, r_opt, K_opt, t0, t_akhir, h=0.1)
_, P01 = ambil_tahunan(t01, P01_halus, 0.1)

# SciPy solve_ivp
sol = solve_ivp(lambda t, P: laju_logistik(P, r_opt, K_opt), (t0, t_akhir), [P0],
                t_eval=tahun, rtol=1e-8)
P_scipy = sol.y[0]

P_eksak = solusi_eksak(tahun, P0, r_opt, K_opt, t0)

perbandingan = pd.DataFrame({
    "Metode": ["Euler h = 1", "Euler h = 0.1", "SciPy solve_ivp"],
    "Galat maks vs eksak": [np.max(np.abs(P1 - P_eksak)), np.max(np.abs(P01 - P_eksak)),
                            np.max(np.abs(P_scipy - P_eksak))],
    "RMSE vs data asli": [evaluasi(data_asli, P1)[0], evaluasi(data_asli, P01)[0],
                          evaluasi(data_asli, P_scipy)[0]],
})
perbandingan.round(4)

In [ ]:
plt.figure()
plt.scatter(tahun, data_asli, color="black", label="Data asli", zorder=5)
plt.plot(t1, P1, "o-", color="red", label="Euler h = 1")
plt.plot(t01, P01_halus, color="blue", label="Euler h = 0.1")
plt.plot(tahun, P_eksak, "--", color="green", label="Solusi eksak")
plt.title(f"Pengaruh Step Size h (r = {r_opt:.3f}, K = {K_opt:.1f})")
plt.xlabel("Tahun"); plt.ylabel("Pengguna internet (% penduduk)")
plt.legend(); plt.show()

### 16.1 Kalibrasi ulang dengan h = 0,1
Parameter di atas dikalibrasi untuk $h=1$. Untuk perbandingan yang adil, kita kalibrasi ulang $r$ dan $K$ khusus untuk $h=0,1$ (10 langkah per tahun, nilai dicatat setiap tahun bulat).

In [ ]:
P_grid = np.full(R.shape, P0)
total_galat2 = np.zeros(R.shape)
h = 0.1
for i in range(1, len(data_asli)):
    for _ in range(10):                     # 10 langkah kecil = 1 tahun
        P_grid = P_grid + h * R * P_grid * (1 - P_grid / KK)
    total_galat2 += (P_grid - data_asli[i]) ** 2

rmse_grid01 = np.sqrt(total_galat2 / len(data_asli))
idx01 = np.unravel_index(np.argmin(rmse_grid01), rmse_grid01.shape)
r_opt01, K_opt01 = R[idx01], KK[idx01]

print(f"h = 1   → r = {r_opt:.3f}, K = {K_opt:.1f}, RMSE = {rmse_grid[idx]:.3f}")
print(f"h = 0.1 → r = {r_opt01:.3f}, K = {K_opt01:.1f}, RMSE = {rmse_grid01[idx01]:.3f}")

**Interpretasi perbandingan step size:**
1. **Terhadap solusi eksak**, Euler $h=0{,}1$ jauh lebih akurat: galat maksimum turun dari **10,06** menjadi **1,03** poin persen (sekitar 10 kali lebih kecil, sesuai sifat Euler orde satu). SciPy `solve_ivp` hampir tanpa galat.
2. **Terhadap data asli**, dengan $r$ dan $K$ yang sama, $h=1$ justru paling cocok (RMSE 1,73), sedangkan $h=0{,}1$ dan SciPy RMSE-nya 5,57 dan 6,14. Ini terjadi karena $r$ dan $K$ tadi **dikalibrasi memakai $h=1$**, sehingga galat numerik Euler $h=1$ ikut "terserap" ke dalam nilai parameter.
3. Setelah **kalibrasi ulang untuk $h=0{,}1$**, diperoleh $r=0{,}240$, $K=100{,}0$, RMSE 1,883 — hampir sama baiknya. Namun $K$ menempel di batas atas 100, yang lagi-lagi menunjukkan $K$ belum bisa ditentukan dari data.

**Kesimpulan:** nilai parameter hasil kalibrasi **bergantung pada step size**. Model baseline memakai $h=1$ karena data tahunan dan RMSE-nya terkecil, tetapi saat menyebut nilai $r$ harus dicatat bahwa itu adalah $r$ untuk Euler $h=1$. Untuk mendekati solusi persamaan diferensial yang sebenarnya, gunakan $h$ kecil (misalnya 0,1) beserta parameter yang dikalibrasi ulang.

---
# TAHAP 5 — Eksperimen Skenario (What-If Analysis)

Kita ubah parameter input dan melihat dampaknya terhadap Stock $P$. Simulasi diperpanjang sampai **2040** agar perilaku jangka panjang terlihat. Semua skenario memakai Euler $h=1$ dan nilai awal yang sama ($P_0$ tahun 2000).

| Skenario | Perubahan | Pertanyaan What-If |
|---|---|---|
| Baseline | $r = 0{,}261$, $K = 96{,}9$ (hasil kalibrasi) | — |
| Skenario 1 | $K$ diturunkan menjadi **80%** | Bagaimana jika batas jenuh pengguna internet lebih rendah (misalnya karena sebagian penduduk tidak terjangkau infrastruktur)? |
| Skenario 2 | $r$ diturunkan **50%** | Bagaimana jika laju adopsi internet hanya setengahnya (misalnya pembangunan jaringan dan penetrasi smartphone tertunda)? |

**Alasan pemilihan:** Skenario 1 menguji parameter $K$ yang hasil kalibrasinya paling tidak pasti (lihat peta RMSE). Skenario 2 menguji kekuatan loop penguat (R) yang dikendalikan oleh $r$.

In [ ]:
TAHUN_PROYEKSI = 2040

skenario = {
    "Baseline":                    {"r": r_opt,       "K": K_opt},
    "Skenario 1: K = 80%":         {"r": r_opt,       "K": 80.0},
    "Skenario 2: r turun 50%":     {"r": r_opt * 0.5, "K": K_opt},
}

hasil_whatif = pd.DataFrame({"Tahun": np.arange(t0, TAHUN_PROYEKSI + 1)})
for nama, p in skenario.items():
    _, P_s = euler_logistik(P0, p["r"], p["K"], t0, TAHUN_PROYEKSI, h=1)
    hasil_whatif[nama] = P_s

hasil_whatif.to_csv("data/hasil_whatif.csv", index=False)
print("Hasil What-If disimpan ke data/hasil_whatif.csv")
hasil_whatif.round(2).iloc[::5]   # tampilkan setiap 5 tahun

## 17. Grafik Baseline vs Skenario

In [ ]:
warna = {"Baseline": "red", "Skenario 1: K = 80%": "purple", "Skenario 2: r turun 50%": "green"}

plt.figure(figsize=(11, 6))
plt.scatter(tahun, data_asli, color="black", zorder=5, label="Data asli 2000–2024")
for nama in skenario:
    plt.plot(hasil_whatif["Tahun"], hasil_whatif[nama], color=warna[nama], linewidth=2,
             label=f"{nama} (r={skenario[nama]['r']:.3f}, K={skenario[nama]['K']:.1f})")
    plt.axhline(skenario[nama]["K"], color=warna[nama], linestyle=":", linewidth=1)
plt.axvspan(t_akhir, TAHUN_PROYEKSI, color="gray", alpha=0.1, label="Periode proyeksi")
plt.title("Eksperimen What-If: Baseline vs Skenario")
plt.xlabel("Tahun"); plt.ylabel("Pengguna internet (% penduduk)")
plt.legend(loc="upper left"); plt.show()

**Interpretasi grafik What-If:**
- **Baseline** mengikuti data 2000–2024, lalu terus naik dan mendatar mendekati 96,9%: sekitar 92,3% pada 2030 dan 96,7% pada 2040.
- **Skenario 1 (K = 80%)** hampir berimpit dengan baseline sampai ±2015, karena saat $P$ masih kecil faktor pembatas $(1-P/K)$ hampir sama dengan 1. Perbedaan baru terlihat setelah $P$ mendekati 80%: kurva mendatar lebih cepat dan lebih rendah, yaitu 77,0% pada 2030 dan 79,9% pada 2040.
- **Skenario 2 (r turun 50%)** membuat seluruh kurva **bergeser jauh ke kanan**: tahun 2024 baru 15,2% (data asli 72,8%), dan batas 50% baru tercapai pada **2038**, terlambat 18 tahun dibanding baseline (2020). Batas akhirnya tetap 96,9% karena $K$ tidak berubah — yang berubah hanya **kecepatan** menuju batas tersebut.

## 18. Perilaku Laju Pertumbuhan (Flow)
Grafik laju $dP/dt$ menunjukkan kapan loop penguat (R) dominan (laju naik) dan kapan loop penyeimbang (B) mengambil alih (laju turun). Puncak laju = **titik belok**.

In [ ]:
plt.figure()
for nama, p in skenario.items():
    laju = laju_logistik(hasil_whatif[nama], p["r"], p["K"])
    plt.plot(hasil_whatif["Tahun"], laju, color=warna[nama], linewidth=2, label=nama)
plt.title("Laju Pertumbuhan Pengguna Internet (dP/dt) per Skenario")
plt.xlabel("Tahun"); plt.ylabel("Pengguna baru (poin persen per tahun)")
plt.legend(); plt.show()

**Interpretasi laju:** Setiap kurva laju berbentuk "bukit". Sisi naik = loop penguat (R) dominan; puncak = titik belok ($P \approx K/2$); sisi turun = loop penyeimbang (B) dominan. Baseline mencapai puncak 6,31 poin per tahun pada 2019. Menurunkan $K$ membuat bukit **lebih rendah** (5,20 poin per tahun) dengan waktu puncak hampir sama (2019). Menurunkan $r$ membuat bukit **lebih rendah sekaligus lebih lebar dan bergeser jauh ke kanan** (puncak 3,16 poin per tahun pada 2037).

## 19. Ringkasan Perbandingan Skenario

In [ ]:
def tahun_pertama(t, P, batas):
    """Tahun pertama ketika P mencapai/melewati batas (None jika belum tercapai)."""
    m = np.where(P >= batas)[0]
    return int(t[m[0]]) if len(m) else None

ringkasan = []
for nama, p in skenario.items():
    P_s = hasil_whatif[nama].values
    t_s = hasil_whatif["Tahun"].values
    laju = laju_logistik(P_s, p["r"], p["K"])
    ringkasan.append({
        "Skenario": nama,
        "r": round(p["r"], 3),
        "K (%)": round(p["K"], 1),
        "P 2024 (%)": round(P_s[t_s == 2024][0], 2),
        "P 2030 (%)": round(P_s[t_s == 2030][0], 2),
        "P 2040 (%)": round(P_s[t_s == 2040][0], 2),
        "Tahun capai 50%": tahun_pertama(t_s, P_s, 50),
        "Laju puncak (poin/thn)": round(laju.max(), 2),
        "Tahun laju puncak": int(t_s[np.argmax(laju)]),
        "RMSE vs data 2000–2024": round(evaluasi(data_asli, P_s[:len(data_asli)])[0], 3),
    })

ringkasan = pd.DataFrame(ringkasan)
ringkasan

## 20. Kesimpulan Eksperimen What-If

| Aspek | Skenario 1: K = 80% | Skenario 2: r turun 50% |
|---|---|---|
| Parameter diubah | Kapasitas maksimum $K$: 96,9 → 80 | Laju pertumbuhan $r$: 0,261 → 0,131 |
| Dampak utama | Mengubah **seberapa tinggi** kurva berhenti | Mengubah **seberapa cepat** kurva naik |
| Perilaku awal | Hampir sama dengan baseline | Jauh lebih lambat sejak awal |
| P tahun 2040 | 79,85% (baseline 96,66%) | 57,29% |
| Tahun mencapai 50% | 2021 (baseline 2020) | 2038 |
| RMSE terhadap data 2000–2024 | 3,986 | 27,029 |

1. **$r$ mengendalikan loop penguat (R)** → menentukan kecepatan adopsi dan waktu titik belok.
2. **$K$ mengendalikan loop penyeimbang (B)** → menentukan batas akhir dan kapan pertumbuhan mulai melambat.
3. Skenario 2 jelas **tidak sesuai** dengan kenyataan (RMSE 27,0), artinya laju adopsi Indonesia yang sebenarnya memang jauh lebih cepat.
4. Skenario 1 masih **cukup dekat** dengan data (RMSE 4,0, baseline 1,7). Ini menegaskan temuan di peta RMSE: data 2000–2024 belum bisa memastikan nilai $K$. Proyeksi setelah 2024 karena itu bersifat **skenario**, bukan ramalan pasti — angka jenuh 80% maupun 97% sama-sama masih mungkin, dan data beberapa tahun ke depan yang akan menentukan.